# Detección de defectos en infraestructuras (`Corrosion-bi3q3`) con RetinaNet v2

**Asignatura:** Visión por Computador — Grado en Inteligencia Artificial (USC, 2026/2027)  
**Dataset asignado:** `Francesco/corrosion-bi3q3` (Roboflow-100)  
**Detector analizado:** RetinaNet v2 (`retinanet_resnet50_fpn_v2`, pesos `COCO_V1` de `torchvision`)

Este notebook sirve para **visualizar y comparar resultados**, no para entrenar modelos:
1. Descripción del dataset y de **RetinaNet v2**.
2. Metodología y curvas del **baseline** (pérdidas y mAP de validación).
3. **Estrategias de mejora**: hipótesis y experimentos propuestos.
4. **Comparación cuantitativa sobre validación** para elegir modelos sin utilizar test.
5. **Análisis cualitativo** de aciertos y errores en validación.
6. Preparación del **ensemble con Faster R-CNN v2**.
7. **Evaluación final en test**, una vez elegidos los modelos.



In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.patches import Rectangle
from IPython.display import display

# Localizar la raíz del proyecto tanto desde la raíz como desde notebooks/.
RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
if not (RAIZ / "src").exists():
    raise FileNotFoundError("Abre el notebook desde la raíz del repositorio o desde notebooks/.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.dataset import CorrosionDataset, NOMBRES_CLASES, PARTICIONES
from src.evaluar import cargar_predicciones, _iou_entre_cajas

DIR_RESULTADOS = RAIZ / "results" / "slippage_validation_split"
DIR_FIGURAS = DIR_RESULTADOS / "figuras_retinanet"
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

COLORES_CLASE = {
    1: "#1f77b4",  # Slippage
    2: "#d62728",  # corrosion
    3: "#ff7f0e",  # crack
}

print(f"Raíz del proyecto: {RAIZ}")
print(f"Figuras: {DIR_FIGURAS}")
print(f"CUDA disponible: {torch.cuda.is_available()} (no es necesaria para analizar resultados guardados)")


---
## 1. Descripción del dataset (`Corrosion-bi3q3`) y del modelo RetinaNet v2

Por indicación del profesor, se trasladaron 20 imágenes completas con `Slippage` del entrenamiento original a validación. Ambos detectores usan la misma división corregida. Hay que reentrenar ambos modelos para obtener resultados con esta división. [Manifiesto](../configs/particiones/slippage_validation_split.json)

### Particularidades del dataset
- **Resolución:** imágenes de `640 × 640` píxeles. Particiones: `train=820`, `validation=125`, `test=304`; hay imágenes sin objetos anotados (`37 / 5 / 13`).
- **Clases:** `1=Slippage`, `2=corrosion`, `3=crack`; el índice `0` queda reservado y no hay anotaciones de la categoría original `corrosion-0`.
- **Desequilibrio de clases.** En `train` hay 499 anotaciones de `Slippage`, 1.175 de `corrosion` y 1.388 de `crack`. En `validation` hay 49, 165 y 258, respectivamente.
- **Objetos pequeños:** aproximadamente el `85,2 %` de las anotaciones tienen un área inferior a `32²=1024 px²`. Es una posible motivación para probar mayor resolución o anchors adaptados.

### Arquitectura: RetinaNet v2 (`ResNet-50 + FPN v2`)
- Detector **de una etapa (*one-stage*)**: predice clases y cajas directamente sobre una pirámide multiescala de características, sin la fase de propuestas/RoI de Faster R-CNN.
- **Backbone ResNet-50 + FPN** y cabeza de clasificación/regresión; parte de los pesos `RetinaNet_ResNet50_FPN_V2_Weights.COCO_V1` de torchvision.
- **Clasificación por ancla:** utiliza una clasificación sigmoide con pérdida focal, a diferencia del *softmax* de Faster R-CNN. En nuestra adaptación se reservan 4 índices (`0` sin anotaciones, `1..3` defectos) y se generan **9 × 4 = 36 logits** por posición de la cabeza de clasificación.
- Se conserva la cabeza de regresión, GroupNorm y los anchors por defecto para establecer un baseline comparable. Las modificaciones de anchors o resolución se plantearán como experimentos posteriores.

Las figuras siguientes caracterizan el dataset; no dependen de ningún entrenamiento.


In [ ]:
# Análisis cuantitativo de clases, tamaños y relaciones de aspecto (motivación de mejoras)

ruta_dataset = RAIZ / "data" / "corrosion-bi3q3"
if not ruta_dataset.exists():
    print("Dataset no encontrado. Ejecuta: uv run python scripts/descargar_datos.py")
else:
    datos_hf = {particion: CorrosionDataset(particion).datos for particion in PARTICIONES}

    filas_resumen = []
    ratios_por_clase = {nombre: [] for nombre in NOMBRES_CLASES.values()}
    areas_por_clase = {nombre: [] for nombre in NOMBRES_CLASES.values()}

    for particion, datos in datos_hf.items():
        cont = Counter()
        vacias = 0
        for objetos in datos["objects"]:
            cajas = objetos["bbox"]
            cats = objetos["category"]
            if len(cajas) == 0:
                vacias += 1
            cont.update(cats)
            if particion == "train":
                for (x, y, w, h), cat in zip(cajas, cats):
                    nombre = NOMBRES_CLASES[cat]
                    areas_por_clase[nombre].append(w * h)
                    ratios_por_clase[nombre].append(w / max(h, 1e-6))

        filas_resumen.append({
            "Partición": particion,
            "Imágenes": len(datos),
            "Sin objetos": vacias,
            "Slippage": cont.get(1, 0),
            "corrosion": cont.get(2, 0),
            "crack": cont.get(3, 0),
            "Total objetos": sum(cont.values()),
        })

    df_dataset = pd.DataFrame(filas_resumen)
    display(df_dataset)

    # Gráfica de distribución de áreas y relaciones de aspecto por clase en train
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    for nombre, areas in areas_por_clase.items():
        axes[0].hist(np.log10(np.asarray(areas) + 1), bins=30, alpha=0.55, label=f"{nombre} (med={np.median(areas):.0f} px²)")
    axes[0].axvline(np.log10(32**2), color="black", linestyle="--", linewidth=1.2, label="Umbral COCO small (32²)")
    axes[0].set_xlabel("log10(área en px² + 1)")
    axes[0].set_ylabel("Número de cajas")
    axes[0].set_title("Distribución de tamaños por clase (train)")
    axes[0].legend(fontsize=8)

    for nombre, ratios in ratios_por_clase.items():
        axes[1].hist(np.log2(np.asarray(ratios)), bins=30, alpha=0.55, label=nombre)
    axes[1].axvline(np.log2(0.5), color="black", linestyle=":", linewidth=1.2, label="Anchors defecto (0.5 y 2.0)")
    axes[1].axvline(np.log2(2.0), color="black", linestyle=":", linewidth=1.2)
    axes[1].set_xlabel("log2(ancho / alto)")
    axes[1].set_ylabel("Número de cajas")
    axes[1].set_title("Relaciones de aspecto por clase vs anchors por defecto")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(DIR_FIGURAS / "01_eda_tamanos_aspecto_retinanet.png", bbox_inches="tight")
    plt.show()


### Análisis de los tamaños y las relaciones de aspecto

La distribución de áreas muestra un claro predominio de objetos pequeños, especialmente en la clase `crack`, cuya mediana es de **165 px²**, frente a los **333 px²** de `corrosion` y los **314 px²** de `Slippage`. Gran parte de las anotaciones se encuentran por debajo del umbral de objetos pequeños de COCO (32 × 32 píxeles).

En cuanto a las relaciones de aspecto, las grietas presentan una distribución especialmente alargada, con numerosas cajas cuya relación ancho/alto es inferior a 0,5. Esto sugiere que los anchors predeterminados podrían no ajustarse de forma óptima a la geometría de algunos defectos.

**Implicaciones para RetinaNet:** estos resultados justifican experimentar posteriormente con una mayor resolución de entrada y con tamaños y relaciones de aspecto de anchors adaptados al dataset. Será necesario comprobar mediante las métricas de validación si estas modificaciones mejoran realmente la detección.

---
## 2. Metodología y resultados del baseline

### Configuración (`configs/retinanet.yaml`)
- **Modelo:** `retinanet_resnet50_fpn_v2` preentrenado en COCO, con la última convolución de clasificación adaptada a 4 índices (`0..3`), `capas_entrenables_backbone=3` y resolución interna `min_size=800`, `max_size=1333`.
- **Entrenamiento:** 12 épocas, AdamW (`lr=1e-4`, `weight_decay=1e-4`), `batch_size=2`, acumulación de gradientes (`4 pasos`), semilla `42` y AMP si hay CUDA. El resto de los parámetros se toma del YAML.
- **Selección:** se guarda el mejor checkpoint según **mAP@[0.50:0.95] en `validation`**. **No se utiliza `test` para elegir estrategias.**

Para entrenar fuera de este notebook:

```bash
uv run --locked python -m src.entrenar --config configs/retinanet.yaml --sin-test
```

Esto genera `results/slippage_validation_split/retinanet/historial.json`, `mejor_modelo.pth` y `ultimo_modelo.pth` (entre otros archivos). Después, **si se quieren curvas PR, AP por clase y latencia en validación**, ejecutar:

```bash
uv run --locked python -m src.evaluar --checkpoint results/slippage_validation_split/retinanet/mejor_modelo.pth --particion validation --guardar results/slippage_validation_split/retinanet/metricas_validation.json --guardar-preds results/slippage_validation_split/retinanet/predicciones_validation.json
```

El notebook lee estos archivos cuando existan, sin volver a entrenar ni cargar el modelo en GPU.


In [ ]:
# Registro de experimentos: nombre legible -> carpeta de resultados.
# Los directorios E1-E3 son propuestas: todavía hay que crear sus YAML y entrenarlos.
NOMBRE_BASELINE = "Baseline (RetinaNet v2)"
EXPERIMENTOS = {
    NOMBRE_BASELINE: {
        "dir": DIR_RESULTADOS / "retinanet",
        "categoria": "Baseline",
        "hipotesis": "RetinaNet v2 COCO_V1, AdamW, 800 px y anchors por defecto.",
    },
    "E1: Aumento de datos (propuesto)": {
        "dir": DIR_RESULTADOS / "retinanet_exp01_aumentos",
        "categoria": "Aumentos",
        "hipotesis": "Volteos y variaciones de color para mejorar la generalización.",
    },
    "E2: Resolución (propuesto)": {
        "dir": DIR_RESULTADOS / "retinanet_exp02_resolucion",
        "categoria": "Resolución",
        "hipotesis": "Evaluar mayor resolución interna para objetos pequeños.",
    },
    "E3: Optimizador (propuesto)": {
        "dir": DIR_RESULTADOS / "retinanet_exp03_sgd",
        "categoria": "Optimización",
        "hipotesis": "Comparar SGD + momentum frente a AdamW.",
    },
}


def leer_json(ruta: Path) -> dict | list | None:
    if not ruta.exists():
        return None
    with ruta.open(encoding="utf-8") as archivo:
        return json.load(archivo)


def cargar_historial_experimento(dir_exp: Path) -> list[dict] | None:
    datos = leer_json(dir_exp / "historial.json")
    return datos if isinstance(datos, list) else None


def cargar_metricas_experimento(dir_exp: Path, particion: str = "validation") -> dict | None:
    """Carga métricas detalladas o, en validación, las del mejor registro del historial."""
    ruta = dir_exp / f"metricas_{particion}.json"
    datos = leer_json(ruta)
    if isinstance(datos, dict):
        if particion in datos and isinstance(datos[particion], dict):
            return datos[particion]  # train.py envuelve así metricas_test.json
        if "map" in datos:
            return datos  # evaluar.py guarda directamente el diccionario
    if particion == "validation":
        historial = cargar_historial_experimento(dir_exp)
        if historial:
            registros = [h.get("metricas_val", {}) for h in historial]
            registros = [m for m in registros if isinstance(m, dict) and "map" in m]
            if registros:
                return max(registros, key=lambda m: m["map"])
    return None


# Curvas de entrenamiento del baseline: claves propias de RetinaNet.
hist_base = cargar_historial_experimento(EXPERIMENTOS[NOMBRE_BASELINE]["dir"])
if hist_base:
    epocas = [h["epoca"] for h in hist_base]
    perdida_total = [h["perdidas_train"].get("total", np.nan) for h in hist_base]
    perdida_cls = [h["perdidas_train"].get("classification", np.nan) for h in hist_base]
    perdida_cajas = [h["perdidas_train"].get("bbox_regression", np.nan) for h in hist_base]
    val_map = [h["metricas_val"]["map"] for h in hist_base]
    val_map50 = [h["metricas_val"]["map_50"] for h in hist_base]
    val_small = [h["metricas_val"].get("map_small", np.nan) for h in hist_base]

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))
    axes[0].plot(epocas, perdida_total, marker="o", label="Pérdida total")
    axes[0].plot(epocas, perdida_cls, marker="s", linestyle="--", label="Clasificación (focal)")
    axes[0].plot(epocas, perdida_cajas, marker="^", linestyle="--", label="Regresión de cajas")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("Pérdida (train)")
    axes[0].set_title("Pérdidas de entrenamiento — RetinaNet v2")
    axes[0].legend()

    axes[1].plot(epocas, val_map50, marker="o", label="mAP@0.50")
    axes[1].plot(epocas, val_map, marker="s", label="mAP@[0.50:0.95]")
    axes[1].plot(epocas, val_small, marker="^", linestyle="--", label="mAP_small")
    mejor_idx = int(np.nanargmax(val_map))
    axes[1].axvline(epocas[mejor_idx], color="gray", linestyle=":", label=f"Mejor época ({epocas[mejor_idx]})")
    axes[1].set_xlabel("Época")
    axes[1].set_ylabel("mAP (validation)")
    axes[1].set_title("Evolución de mAP — RetinaNet v2")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(DIR_FIGURAS / "02_baseline_curvas_entrenamiento.png", bbox_inches="tight")
    plt.show()
else:
    print("Aún no existe results/slippage_validation_split/retinanet/historial.json. Ejecuta primero el baseline.")


In [ ]:
# Curvas precisión-recall y resumen del baseline en VALIDACIÓN.
met_base = cargar_metricas_experimento(EXPERIMENTOS[NOMBRE_BASELINE]["dir"], "validation")

if met_base:
    print("=== RetinaNet baseline: validación ===")
    print(f"mAP@[0.50:0.95]: {met_base['map']:.4f} | mAP@0.50: {met_base['map_50']:.4f}")
    if "map_small" in met_base:
        print(f"mAP_small: {met_base['map_small']:.4f}")
    if "tiempos" in met_base:
        t = met_base["tiempos"]
        print(f"Latencia: {t['ms_por_imagen']:.2f} ms/img ({t['fps']:.2f} FPS)")

    if "curvas_pr_50" in met_base:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
        for ax, clave, titulo in [
            (axes[0], "curvas_pr_50", "PR por clase (IoU = 0.50)"),
            (axes[1], "curvas_pr_75", "PR por clase (IoU = 0.75)"),
        ]:
            curvas = met_base.get(clave, {})
            for id_c, nombre in NOMBRES_CLASES.items():
                c = curvas.get(nombre)
                if c and c.get("n_gt", 0) > 0 and len(c.get("recall", [])) > 0:
                    ax.plot(c["recall"], c["precision_interp"], linewidth=2,
                            color=COLORES_CLASE[id_c], label=f"{nombre} (AP={c['ap']:.3f})")
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1.05)
            ax.set_xlabel("Recall")
            ax.set_title(titulo)
            if ax.lines:
                ax.legend(loc="lower left")
        axes[0].set_ylabel("Precision interpolada")
        plt.tight_layout()
        plt.savefig(DIR_FIGURAS / "03_baseline_curvas_pr_validation.png", bbox_inches="tight")
        plt.show()
    else:
        print("Para obtener curvas PR y tiempos, evalúa el checkpoint sobre validation:")
        print("uv run --locked python -m src.evaluar --checkpoint results/slippage_validation_split/retinanet/mejor_modelo.pth "
              "--particion validation --guardar results/slippage_validation_split/retinanet/metricas_validation.json "
              "--guardar-preds results/slippage_validation_split/retinanet/predicciones_validation.json")
else:
    print("Todavía no se encontraron métricas de validación para RetinaNet.")


---
## 3. Estrategias de mejora propuestas (guion de la práctica)

Estas son **hipótesis a comprobar**, no resultados obtenidos ni experimentos ya implementados. Primero analiza el baseline y después elige qué merece la pena ejecutar.

| Estrategia | Categoría | Hipótesis y modificación | Estado |
|---|---|---|---|
| **E1: Aumento de datos** | Data augmentation | Volteos, cambios de iluminación y recortes controlados podrían mejorar la generalización. | Pendiente de configurar y entrenar. |
| **E2: Mayor resolución** | Hiperparámetros | Una resolución interna superior a 800 px podría ayudar a las muchas cajas pequeñas; comprobar coste en memoria y latencia. | Pendiente de crear YAML y entrenar. |
| **E3: SGD frente a AdamW** | Optimización | Comparar otra combinación de optimizador y scheduler manteniendo fijo lo demás. | El pipeline admite SGD; falta configurar y entrenar. |
| **E4: Anchors adaptados** | Arquitectura | Modificar tamaños/ratios podría beneficiar grietas finas, pero requiere adaptar y validar la cabeza de RetinaNet. | No implementado todavía. |
| **E5: TTA / WBF** | Posprocesado | Fusionar predicciones de imágenes volteadas podría mejorar recall sin reentrenar. | Hay utilidades genéricas; verificar la integración con RetinaNet antes de usarlo. |

Los directorios propuestos para E1–E3 en `EXPERIMENTOS` se pueden ajustar al nombre real de cada ejecución.

**Precaución:** el barrido NMS existente para Faster R-CNN modifica `roi_heads`, un componente que RetinaNet no tiene. No reutilizar ese subcomando directamente sin adaptarlo a su arquitectura.

**Protocolo:** escoger las mejoras con `validation` y ejecutar la evaluación de `test` solo cuando se haya fijado la selección final.


---
## 4. Evolución y comparativa cuantitativa de resultados (validación)

Esta sección lee `historial.json` y, cuando existan, `metricas_validation.json` de cada experimento. Compara:
1. mAP@[0.50:0.95], mAP@0.50, mAP_small y mejora respecto al baseline.
2. AP por clase para `Slippage`, `corrosion` y `crack`.
3. Latencia medida en la evaluación detallada, si está disponible.
4. Curvas PR y distribución aproximada de errores, si se han guardado las métricas detalladas.

La selección del mejor modelo se hace **exclusivamente con mAP de validación**. No se utiliza `test` para establecer el ranking.


In [ ]:
filas_comp = []
metricas_por_exp = {}

for nombre, meta in EXPERIMENTOS.items():
    m = cargar_metricas_experimento(meta["dir"], "validation")
    if not m:
        continue
    metricas_por_exp[nombre] = m
    ap_clase = m.get("map_por_clase", {})
    tiempos = m.get("tiempos", {})
    filas_comp.append({
        "Experimento": nombre,
        "Categoría": meta["categoria"],
        "mAP@0.50:0.95": m.get("map", np.nan),
        "mAP@0.50": m.get("map_50", np.nan),
        "mAP_small": m.get("map_small", np.nan),
        "AP Slippage": ap_clase.get("Slippage", np.nan),
        "AP corrosion": ap_clase.get("corrosion", np.nan),
        "AP crack": ap_clase.get("crack", np.nan),
        "ms/img": tiempos.get("ms_por_imagen", np.nan),
        "FPS": tiempos.get("fps", np.nan),
    })

if filas_comp:
    df_comp = pd.DataFrame(filas_comp)
    base = metricas_por_exp.get(NOMBRE_BASELINE)
    referencia = base["map"] if base is not None else np.nan
    df_comp.insert(3, "ΔmAP vs Base", df_comp["mAP@0.50:0.95"] - referencia)
    print("Comparativa en validación:")
    display(df_comp.round(4))

    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.7 * len(df_comp) + 1.5)))
    y = np.arange(len(df_comp))
    alto = 0.26
    ax.barh(y - alto, df_comp["mAP@0.50"], height=alto, label="mAP@0.50")
    ax.barh(y, df_comp["mAP@0.50:0.95"], height=alto, label="mAP@[0.50:0.95]")
    ax.barh(y + alto, df_comp["mAP_small"], height=alto, label="mAP_small")
    ax.set_yticks(y)
    ax.set_yticklabels(df_comp["Experimento"])
    ax.invert_yaxis()
    ax.set_xlabel("Puntuación COCO (validation)")
    ax.set_title("RetinaNet v2 — estrategias de mejora en validación")
    ax.legend()
    plt.tight_layout()
    plt.savefig(DIR_FIGURAS / "04_comparativa_validacion.png", bbox_inches="tight")
    plt.show()

    df_lat = df_comp.dropna(subset=["ms/img"])
    if not df_lat.empty:
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        ax.scatter(df_lat["ms/img"], df_lat["mAP@0.50:0.95"], s=90)
        for _, fila in df_lat.iterrows():
            ax.annotate(fila["Experimento"], (fila["ms/img"], fila["mAP@0.50:0.95"]),
                        xytext=(6, 4), textcoords="offset points", fontsize=8)
        ax.set_xlabel("Tiempo de inferencia (ms/imagen)")
        ax.set_ylabel("mAP@[0.50:0.95] (validation)")
        ax.set_title("Precisión frente a latencia — RetinaNet v2")
        plt.tight_layout()
        plt.savefig(DIR_FIGURAS / "05_tradeoff_latencia_validation.png", bbox_inches="tight")
        plt.show()
else:
    print("Aún no hay métricas de validación. Ejecuta el baseline y vuelve a abrir el notebook.")


In [ ]:
# Comparación PR y errores aproximados entre baseline y mejor configuración,
# seleccionada EXCLUSIVAMENTE con validación.
if metricas_por_exp:
    nombre_mejor = max(metricas_por_exp, key=lambda k: metricas_por_exp[k].get("map", -1))
    print(f"Mejor configuración en validation: {nombre_mejor} "
          f"(mAP={metricas_por_exp[nombre_mejor]['map']:.4f})")

    claves_err = ["TP", "Loc", "Cls", "Dup", "Bkg", "Miss"]
    exps_con_err = [n for n, m in metricas_por_exp.items() if "analisis_errores_05" in m]
    if exps_con_err:
        fig, ax = plt.subplots(figsize=(10.5, max(3.5, 0.6 * len(exps_con_err) + 1.2)))
        y = np.arange(len(exps_con_err))
        izquierda = np.zeros(len(exps_con_err))
        for clave in claves_err:
            valores = np.array([metricas_por_exp[n]["analisis_errores_05"].get(clave, 0)
                                for n in exps_con_err])
            ax.barh(y, valores, left=izquierda, label=clave)
            izquierda += valores
        ax.set_yticks(y)
        ax.set_yticklabels(exps_con_err)
        ax.invert_yaxis()
        ax.set_xlabel("Número de detecciones/instancias")
        ax.set_title("Errores aproximados (estilo TIDE, no análisis TIDE oficial)")
        ax.legend(ncol=6, loc="upper center", bbox_to_anchor=(0.5, -0.15))
        plt.tight_layout()
        plt.savefig(DIR_FIGURAS / "06_errores_validacion.png", bbox_inches="tight")
        plt.show()
    else:
        print("Para comparar tipos de errores, guarda metricas_validation.json con src.evaluar.")

    if (NOMBRE_BASELINE in metricas_por_exp
            and "curvas_pr_50" in metricas_por_exp[NOMBRE_BASELINE]
            and "curvas_pr_50" in metricas_por_exp[nombre_mejor]):
        curvas_base = metricas_por_exp[NOMBRE_BASELINE]["curvas_pr_50"]
        curvas_mejor = metricas_por_exp[nombre_mejor]["curvas_pr_50"]
        # Mostrar las clases para las que se guardaron curvas con anotaciones.
        clases_disponibles = [nombre for nombre in NOMBRES_CLASES.values()
                              if curvas_base.get(nombre, {}).get("n_gt", 0) > 0]
        if clases_disponibles:
            fig, axes = plt.subplots(1, len(clases_disponibles),
                                     figsize=(5.5 * len(clases_disponibles), 4), squeeze=False,
                                     sharey=True)
            for ax, nombre in zip(axes[0], clases_disponibles):
                b = curvas_base.get(nombre)
                m = curvas_mejor.get(nombre)
                if b and len(b.get("recall", [])) > 0:
                    ax.plot(b["recall"], b["precision_interp"], "--",
                            label=f"Baseline (AP50={b['ap']:.3f})")
                if nombre_mejor != NOMBRE_BASELINE and m and len(m.get("recall", [])) > 0:
                    ax.plot(m["recall"], m["precision_interp"], "-",
                            label=f"Mejor (AP50={m['ap']:.3f})")
                ax.set_title(nombre)
                ax.set_xlabel("Recall")
                ax.set_xlim(0, 1)
                ax.set_ylim(0, 1.05)
                if ax.lines:
                    ax.legend(loc="lower left")
            axes[0, 0].set_ylabel("Precision (IoU=0.50)")
            plt.tight_layout()
            plt.savefig(DIR_FIGURAS / "07_curvas_pr_comparativa_validation.png", bbox_inches="tight")
            plt.show()
    else:
        print("Faltan curvas PR detalladas para comparar baseline y mejor experimento.")
else:
    print("Sin experimentos evaluados todavía.")


---
## 5. Análisis cualitativo: aciertos y errores frecuentes

Usamos, durante el desarrollo, imágenes de **`validation`** y las predicciones guardadas en `predicciones_validation.json`:
- **Cajas verdes discontinuas:** *ground truth* (anotaciones).
- **Cajas sólidas de color:** predicciones de RetinaNet (azul = `Slippage`, rojo = `corrosion`, naranja = `crack`) con puntuación de confianza.
- Seleccionamos imágenes con mayor proporción de aciertos y con más falsos positivos/falsos negativos, usando correspondencias de clase e IoU≥0.5.

Revisad visualmente los falsos positivos aparentes, ya que algunas anotaciones pueden estar incompletas. **No utilices ejemplos de test para elegir la mejor estrategia.**


In [ ]:
def dibujar_detecciones(ax, imagen_tensor, gt, pred, titulo="", score_thr=0.50):
    ax.imshow(imagen_tensor.permute(1, 2, 0).numpy())
    for caja, etq in zip(gt["boxes"].numpy(), gt["labels"].numpy()):
        x1, y1, x2, y2 = caja
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor="#00e676", linewidth=1.8, linestyle="--"))
    for caja, score, etq in zip(pred["boxes"].numpy(), pred["scores"].numpy(), pred["labels"].numpy()):
        if score < score_thr:
            continue
        x1, y1, x2, y2 = caja
        color = COLORES_CLASE.get(int(etq), "yellow")
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                               edgecolor=color, linewidth=1.8))
        ax.text(x1, max(0, y1 - 3), f"{NOMBRES_CLASES.get(int(etq), str(etq))} {score:.2f}",
                color="white", fontsize=7,
                bbox={"facecolor": color, "alpha": 0.75, "pad": 1})
    ax.set_title(titulo, fontsize=9)
    ax.axis("off")


def clasificar_imagenes_cualitativo(ds, preds, score_thr=0.50):
    """Cuenta TP, FP y FN con emparejamiento uno-a-uno, clase e IoU>=0.5."""
    resumenes = []
    for idx in range(len(ds)):
        _, gt = ds[idx]
        pred = preds[idx]
        cajas_gt, clases_gt = gt["boxes"].numpy(), gt["labels"].numpy()
        mask = pred["scores"].numpy() >= score_thr
        cajas_p, clases_p = pred["boxes"].numpy()[mask], pred["labels"].numpy()[mask]
        scores_p = pred["scores"].numpy()[mask]

        usados = set()
        tp = 0
        if len(cajas_p) and len(cajas_gt):
            ious = _iou_entre_cajas(cajas_p, cajas_gt)
            for i in np.argsort(-scores_p):
                validos = [j for j in range(len(cajas_gt))
                           if j not in usados and clases_p[i] == clases_gt[j] and ious[i, j] >= 0.5]
                if validos:
                    j = max(validos, key=lambda j: ious[i, j])
                    usados.add(j)
                    tp += 1
        fn = len(cajas_gt) - tp
        fp = len(cajas_p) - tp
        resumenes.append({"idx": idx, "n_gt": len(cajas_gt), "tp": tp, "fp": fp, "fn": fn})
    return resumenes


# Elegir la mejor configuración SOLO según las métricas de validación.
ruta_preds_mejor = None
nombre_cual = None
for nombre in sorted(metricas_por_exp, key=lambda n: -metricas_por_exp[n].get("map", -1)):
    candidata = EXPERIMENTOS[nombre]["dir"] / "predicciones_validation.json"
    if candidata.exists():
        ruta_preds_mejor = candidata
        nombre_cual = nombre
        break

if ruta_preds_mejor is not None:
    ds_val = CorrosionDataset("validation")
    preds_cual = cargar_predicciones(ruta_preds_mejor)
    if len(preds_cual) != len(ds_val):
        print("Las predicciones guardadas no coinciden con el número de imágenes de validación.")
    else:
        stats = clasificar_imagenes_cualitativo(ds_val, preds_cual, score_thr=0.50)
        aciertos = sorted([s for s in stats if s["n_gt"] >= 1],
                          key=lambda s: (-s["tp"] / max(s["n_gt"], 1), s["fp"] + s["fn"]))[:3]
        fallos = sorted([s for s in stats if s["n_gt"] >= 1],
                        key=lambda s: -(s["fp"] + s["fn"]))[:3]
        fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.5))
        for fila, grupo, tipo in [(0, aciertos, "Acierto"), (1, fallos, "Fallo")]:
            for col, ax in enumerate(axes[fila]):
                if col >= len(grupo):
                    ax.axis("off")
                    continue
                s = grupo[col]
                imagen, gt = ds_val[s["idx"]]
                dibujar_detecciones(
                    ax, imagen, gt, preds_cual[s["idx"]],
                    titulo=f"{tipo} (val #{s['idx']}) | TP={s['tp']}, FP={s['fp']}, FN={s['fn']}",
                )
        fig.suptitle(f"Análisis cualitativo (validation) — {nombre_cual}", fontsize=12)
        plt.tight_layout()
        plt.savefig(DIR_FIGURAS / "08_analisis_cualitativo_validation.png", bbox_inches="tight")
        plt.show()
else:
    print("No se encontró predicciones_validation.json. Genera predicciones con src.evaluar.")


---
## 6. Ensemble RetinaNet v2 + Faster R-CNN v2 (Weighted Boxes Fusion)

Cuando ambos modelos tengan resultados y hayáis decidido sus configuraciones usando **validación**, podéis estudiar la fusión de sus detecciones mediante WBF. El código de `src/postprocesado.py` ya ofrece esta operación, pero es necesario comprobar que las predicciones corresponden **a las mismas imágenes en el mismo orden**.

Una vez fijado el ensemble (incluidos pesos e IoU), la comparación final se puede ejecutar sobre `test`, por ejemplo:

```bash
uv run --locked python -m scripts.evaluar_postprocesado ensemble \
  --preds results/slippage_validation_split/faster_rcnn/predicciones_test.json results/slippage_validation_split/retinanet/predicciones_test.json \
  --pesos 1.0 1.0 \
  --iou-wbf 0.55 \
  --salida results/slippage_validation_split/exp08_ensemble_wbf
```

Ajustad las rutas a los modelos finalmente seleccionados. **No optimicéis los hiperparámetros del ensemble mirando su puntuación de test.**


---
## 7. Evaluación final en `test` (una vez elegidos los modelos)

Esta sección **no participa en la selección de experimentos**. Solo presenta resultados finales si ya los habéis calculado.

Después de escoger los checkpoints con validación, ejecutad una evaluación final para cada modelo seleccionado, por ejemplo:

```bash
uv run --locked python -m src.evaluar \
  --checkpoint results/slippage_validation_split/retinanet/mejor_modelo.pth \
  --particion test \
  --guardar results/slippage_validation_split/retinanet/metricas_test_final.json \
  --guardar-preds results/slippage_validation_split/retinanet/predicciones_test.json
```

Si utilizáis la evaluación automática de `src.entrenar` sin `--sin-test`, el archivo `metricas_test.json` tiene una estructura ligeramente distinta; el lector del notebook reconoce ambas.

In [ ]:
# Mostrar resultados finales de test únicamente cuando existan.
# No elegir aquí qué experimento es el mejor: la elección ya se hizo en validación.
filas_test = []
for nombre, meta in EXPERIMENTOS.items():
    ruta_final = meta["dir"] / "metricas_test_final.json"
    if ruta_final.exists():
        m = leer_json(ruta_final)
    else:
        m = cargar_metricas_experimento(meta["dir"], "test")
    if not isinstance(m, dict) or "map" not in m:
        continue
    ap_clase = m.get("map_por_clase", {})
    filas_test.append({
        "Experimento (test final)": nombre,
        "mAP@0.50:0.95": m.get("map", np.nan),
        "mAP@0.50": m.get("map_50", np.nan),
        "mAP_small": m.get("map_small", np.nan),
        "AP Slippage": ap_clase.get("Slippage", np.nan),
        "AP corrosion": ap_clase.get("corrosion", np.nan),
        "AP crack": ap_clase.get("crack", np.nan),
        "ms/img": m.get("tiempos", {}).get("ms_por_imagen", np.nan),
    })

if filas_test:
    print("Resultados finales en test (no usar para elegir hiperparámetros):")
    display(pd.DataFrame(filas_test).round(4))
else:
    print("Todavía no hay métricas finales de test. Es correcto durante el desarrollo.")